# TradeFlow AI — nb4_eval (Final Evaluation)

**Objective**: Evaluate the fine-tuned LoRA model (`muhammadghiffari/olm-ocr-cipl-v1`) against the Real Documents Ground Truth v5.2.
**Metrics**: Weighted ANLS per field + INSW Flag detection accuracy.


In [ ]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes trl qwen-vl-utils rapidfuzz
!pip install -q pdf2image python-dateutil
!apt-get update -qq && apt-get install -qq poppler-utils


In [ ]:
import os, json, re
from pathlib import Path
import torch
from rapidfuzz import fuzz
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
from kaggle_secrets import UserSecretsClient
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

def calculate_anls(gt_val, pred_val, threshold=0.5):
    if not gt_val and not pred_val: return 1.0
    if not gt_val or not pred_val: return 0.0
    gt_str, pred_str = str(gt_val).lower().strip(), str(pred_val).lower().strip()
    ed    = 1.0 - (fuzz.ratio(gt_str, pred_str) / 100.0)
    score = 1.0 - ed
    return score if score >= threshold else 0.0

def load_image(path_str):
    path_str = str(path_str)
    if path_str.endswith('.pdf'):
        pages = convert_from_path(path_str, dpi=150, first_page=1, last_page=1)
        return pages[0].convert('RGB')
    return Image.open(path_str).convert('RGB')

REAL_DOCS_DIR = Path('/kaggle/input/tradeflow-real-docs')
NB0_INPUT     = Path('/kaggle/input/nb0-real-doc-augmentation')
GT_PATH       = REAL_DOCS_DIR / 'TradeFlow_GroundTruth_v5.2.json'
MANIFEST_PATH = NB0_INPUT / 'dataset' / 'augmented_manifest.json'

BASE_MODEL_ID   = 'allenai/olmOCR-2-7B-1025'
LORA_ADAPTER_ID = 'muhammadghiffari/olm-ocr-cipl-v1'

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
except:
    hf_token = None
    print('Warning: HF_TOKEN not found.')


In [ ]:
print('Memuat Base Model & LoRA Adapter...')
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=hf_token)
qconfig   = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, 
    quantization_config=qconfig, 
    device_map='auto',
    token=hf_token
)
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID, token=hf_token)
model.eval()
print('Model siap untuk evaluasi!')


In [ ]:
EXTRACTION_PROMPT = (
    'Extract all CEISA customs declaration fields from this shipping document. '
    'Return valid JSON with these keys: nomorBl, tglBl (YYYY-MM-DD), '
    'pelabuhan_muat, pelabuhan_bongkar, container_no, beratKotor, '
    'hs_code, namaKapal, voyageNumber.'
)

def predict_document(image_path):
    image = load_image(image_path)
    messages = [
        {'role': 'user', 'content': [
            {'type': 'image'},
            {'type': 'text', 'text': EXTRACTION_PROMPT}
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], 
        images=[image], 
        return_tensors='pt'
    ).to('cuda')
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)
        
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    # Extract JSON from output
    try:
        match = re.search(r'\{.*?\}', output_text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return json.loads(output_text)
    except:
        return {}


In [ ]:
EVAL_FIELDS = {
    'nomorBl':           3.0,
    'tglBl':             2.0,
    'pelabuhan_muat':    1.5,
    'pelabuhan_bongkar': 1.5,
    'container_no':      2.0,
    'beratKotor':        2.5,
    'hs_code':           3.0,
    'namaKapal':         1.5,
    'voyageNumber':      1.0,
}

def find_doc_image(doc_key, real_docs_dir, nb0_dir):
    """Find image for a doc_key.
    Key insight: GT uses 'Hapag_Filled_1' but files are 'Hapag Filled 1.pdf' (spaces).
    Try both underscore and space variants.
    """
    doc_key_spaced = doc_key.replace('_', ' ')  # 'Hapag_Filled_1' -> 'Hapag Filled 1'
    variants = [doc_key, doc_key_spaced, doc_key.lower(), doc_key_spaced.lower()]
    exts = ['.pdf', '.png', '.jpg', '.jpeg', '.tiff', '.tif']

    # Strategy 1: exact filename match with all variants
    for name in variants:
        for ext in exts:
            candidate = real_docs_dir / f'{name}{ext}'
            if candidate.exists():
                return str(candidate)

    # Strategy 2: scan all files, check if doc_key (normalized) is in filename
    doc_key_norm = doc_key.lower().replace('_', ' ')
    for f in real_docs_dir.iterdir():
        if f.suffix.lower() in exts:
            if doc_key_norm in f.stem.lower().replace('_', ' '):
                return str(f)

    # Strategy 3: nb0 augmented manifest
    manifest_path = nb0_dir / 'dataset' / 'augmented_manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        for item in manifest.get('train', []) + manifest.get('test', []):
            if item.get('doc_id') == doc_key:
                raw = str(item.get('path', ''))
                if raw.startswith('./dataset'):
                    p = raw.replace('./dataset', str(nb0_dir / 'dataset'), 1)
                elif raw.startswith('/kaggle'):
                    p = raw
                else:
                    p = str(nb0_dir / 'dataset' / raw.lstrip('/'))
                if Path(p).exists():
                    return p
    return None

def evaluate_all():
    if not GT_PATH.exists():
        print(f'File GT tidak ditemukan: {GT_PATH}')
        return

    gt_data = json.loads(GT_PATH.read_text())
    field_scores   = {f: [] for f in EVAL_FIELDS}
    insw_correct   = []

    print('=== MEMULAI EVALUASI REAL INFERENCE ===\n')

    for doc_key, gt_doc in gt_data.items():
        img_path = find_doc_image(doc_key, REAL_DOCS_DIR, NB0_INPUT)

        if not img_path:
            print(f'--- {doc_key} (SKIP: Image not found)')
            continue

        print(f'--- {doc_key} ({gt_doc.get("carrier", "?")})')
        print(f'    Image: {img_path}')

        pred_json = predict_document(img_path)

        ceisa = gt_doc.get('ceisa_fields', gt_doc)
        insw_gt = gt_doc.get('insw_flag', False)
        hs_pred = str(pred_json.get('hs_code', ''))
        insw_pred = hs_pred.startswith('28')

        for field, weight in EVAL_FIELDS.items():
            gt_val   = ceisa.get('container_no_normalized') if field == 'container_no' and not ceisa.get(field) else ceisa.get(field)
            pred_val = pred_json.get(field, '')
            score    = calculate_anls(str(gt_val or ''), str(pred_val or ''))
            field_scores[field].append(score)
            icon = '\u2705' if score >= 0.85 else '\u274c'
            print(f'  {icon} {field:20}: ANLS={score:.3f}  GT={gt_val!r}  Pred={pred_val!r}')

        insw_match = (insw_pred == insw_gt)
        insw_correct.append(insw_match)
        insw_icon = '\u2705' if insw_match else '\u274c'
        print(f'  {insw_icon} insw_flag          : GT={insw_gt} | Pred={insw_pred}')
        print()

    print('\n' + '='*50)
    print('=== HASIL AKHIR WEIGHTED ANLS SCORE ===')
    print('='*50)

    weighted_scores = []
    for field, weight in EVAL_FIELDS.items():
        scores = field_scores[field]
        if not scores: continue
        avg = np.mean(scores)
        weighted_scores.extend([avg] * int(weight * 2))
        status = '\u2705 LULUS' if avg >= 0.85 else '\u274c GAGAL'
        print(f'{field:22} : {avg:.4f}  {status}')

    if not weighted_scores:
        print('\u26a0\ufe0f  Tidak ada dokumen yang berhasil dievaluasi!')
        return

    insw_accuracy = np.mean(insw_correct) if insw_correct else 0.0
    insw_status   = '\u2705 LULUS' if insw_accuracy >= 0.9 else '\u274c GAGAL'
    print(f'{"INSW Flag Detection":22} : {insw_accuracy:.4f}  {insw_status}')

    final_anls = np.mean(weighted_scores)
    print()
    print(f'WEIGHTED ANLS RATA-RATA: {final_anls:.4f}')
    print(f'INSW Detection Accuracy: {insw_accuracy:.4f}')
    print()
    if final_anls >= 0.85:
        print('\U0001f389 MODEL LULUS NFR-007 (Akurasi >= 85%)! Siap deployment.')
    else:
        print(f'\u26a0\ufe0f  Akurasi {final_anls:.1%} belum mencapai target 85%.')

evaluate_all()
